# 02 · Embeddings and vector stores

An embedding maps text to a vector so that similar meanings land close together.
A vector store keeps those vectors and returns the nearest ones for a query.

The default embedder is lexical (shared words only), so it cannot match synonyms.
Set `EMBED_PROVIDER=openai` to use semantic embeddings and compare the results.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from embeddings import get_embedder, cosine
from vector_store import InMemoryVectorStore

embed = get_embedder()
print("Embedder:", getattr(embed, "__name__", type(embed).__name__))

## 1. Similarity between texts

In [ ]:
pairs = [
    ("The cat sat on the mat.", "A cat was sitting on a mat."),
    ("The cat sat on the mat.", "Quarterly revenue grew by twelve percent."),
    ("refund policy", "How do I get my money back?"),
]
for a, b in pairs:
    print(f"{cosine(embed(a), embed(b)):.3f}  |  {a!r}  vs  {b!r}")

With the lexical embedder, the first pair scores high because the words overlap. The
last pair scores low even though the meaning is close. That gap is what semantic
embedding models close. Run this cell with `EMBED_PROVIDER=openai` to compare.

## 2. Build an index and search it

In [ ]:
store = InMemoryVectorStore()
docs = {
    "doc-1": "Refunds are issued within 30 days with a receipt.",
    "doc-2": "Shipping takes three to five business days.",
    "doc-3": "Our support team answers email within one business day.",
    "doc-4": "Warranty covers manufacturing defects for one year.",
}
for doc_id, text in docs.items():
    store.add(doc_id, text, embed(text),
              metadata={"section": "shipping" if doc_id == "doc-2" else "policies"})
print("Items indexed:", len(store))

for hit in store.search(embed("how long does delivery take"), k=2):
    print(hit["score"], hit["id"], "-", hit["text"])

## 3. Metadata filters narrow the search

In [ ]:
for hit in store.search(embed("policy"), k=3, where={"section": "policies"}):
    print(hit["score"], hit["id"], hit["metadata"])

## 4. Save and reload the index

In [ ]:
import os, tempfile
path = os.path.join(tempfile.mkdtemp(), "index.json")
store.save(path)
reloaded = InMemoryVectorStore.load(path)
print("Reloaded items:", len(reloaded))
same_top = (reloaded.search(embed("refund"), k=1)[0]["id"]
            == store.search(embed("refund"), k=1)[0]["id"])
print("Same top hit after reload:", same_top)

## Production notes

- Use the same embedding model for documents and queries. Changing the model means re-embedding everything.
- Store the chunk text and a source id in metadata, so answers can cite their sources.
- Real vector databases (FAISS, Chroma, pgvector, managed services) add approximate nearest-neighbour indexes, persistence, concurrency, and filtering at scale.
- Check whether your store expects normalized vectors when you use dot product versus cosine.